In [ ]:
import streamlit as st
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langsmith import Client


load_dotenv()

index_name = 'tax-index'
embedding = OpenAIEmbeddings(model="text-embedding-3-large")
database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

llm = ChatOpenAI(model='gpt-4o')
client = Client()
rag_prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)
retriever = database.as_retriever(search_kwargs={"k": 4})

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 그대로 반환해주세요.
    사전: {dictionary}

    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()
tax_chain = dictionary_chain | rag_chain

query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
ai_response = tax_chain.invoke({"question": query})

print(f'AI 응답: {ai_response}')

AI 응답: 연봉 5천만원인 거주자의 소득세는 624만원입니다. 이는 과세표준 5천만원 이하 구간에 해당하며 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)을 통해 계산됩니다.
